Weighting Feature Main Data


In [31]:
# Third-Party Library
import pandas as pd
import numpy as np
import joblib
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [2]:
# Load the preprocessed data
file_path = '../data/raw/preprocessed_data.csv'

df = pd.read_csv(file_path)
df.head()

,id_data,sentiment_label,review,processed_text,processed_tokens
0,1,positive,One of the other reviewers has mentioned that ...,one reviewer mention watch 1 oz episode hook r...,"['one', 'reviewer', 'mention', 'watch', '1', '..."
1,2,positive,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...,"['wonderful', 'little', 'production', 'filming..."
2,3,positive,I thought this was a wonderful way to spend ti...,think wonderful way spend time hot summer week...,"['think', 'wonderful', 'way', 'spend', 'time',..."
3,4,negative,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...,"['basically', 'family', 'little', 'boy', 'jake..."
4,5,positive,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...,"['petter', 'mattei', 'love', 'time', 'money', ..."


In [58]:
# Splitting the dataset into features and target variable

X_text = df['processed_text']
y = df['sentiment_label']

# Splitting the dataset into training and testing sets (70% training, 30% testing)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print(f"Training set size: {len(X_train_text)}")
print(f"Testing set size: {len(X_test_text)}\n")
print(f"Training set target balance: {y_train.value_counts()}\n")
print(f"Testing set target balance: {y_test.value_counts()}\n")
print(f"Total dataset size: {len(df)}")


Training set size: 35000
Testing set size: 15000

Training set target balance: sentiment_label
positive    17500
negative    17500
Name: count, dtype: int64

Testing set target balance: sentiment_label
negative    7500
positive    7500
Name: count, dtype: int64

Total dataset size: 50000


In [59]:
# Vectorize (TF-IDF)
vectorizer = TfidfVectorizer(
    min_df=5,
    max_features=10000,
    ngram_range=(1, 2)
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("Vocabulary :", len(vectorizer.vocabulary_))

Train shape: (35000, 10000)
Test shape : (15000, 10000)
Vocabulary : 10000


In [60]:
# Checking the feature names and their corresponding indices
feature_names = vectorizer.get_feature_names_out()

print("Feature count:", len(feature_names))
print("First 10 features:")
print(feature_names[:10])

print("\nLast 10 features:")
print(feature_names[-10:])

print("Vocabulary size:", len(vectorizer.vocabulary_))

Feature count: 10000
First 10 features:
['00' '00 00' '00_neg' '10' '10 10' '10 line' '10 minute' '10 star'
 '10 year' '100']

Last 10 features:
['youtube' 'zane' 'zellweger' 'zero' 'zero_neg' 'zombie' 'zombie movie'
 'zombie_neg' 'zone' 'zoom']
Vocabulary size: 10000


In [61]:
# Validate TF-IDF values for one document

sample_idx = 0

# Get sample document and its TF-IDF vector
sample_text = X_train_text.iloc[sample_idx]
sample_vector = X_train.getrow(sample_idx)

print("Processed Text:")
print(sample_text)

# Get non-zero TF-IDF features
feature_indices = sample_vector.indices
feature_values = sample_vector.data

sample_features = pd.DataFrame({
    'term': feature_names[feature_indices],
    'tfidf': feature_values
}).sort_values(
    'tfidf',
    ascending=False
)

# Validate whether each feature exists in the original processed text
sample_terms = sample_text.lower()

sample_features['in_text'] = sample_features['term'].apply(
    lambda term: term in sample_text.lower()
)

print("\nTop 20 TF-IDF Features:")
sample_features.head(20)

Processed Text:
want see true thriller rent director_NEG screenwriter_NEG scream_NEG feature_NEG overact_NEG overpay_NEG tv_NEG actor_NEG pass_NEG star_NEG run_NEG mill_NEG special_NEG effect_NEG bonanza_NEG instead get top notch edgy strong violence yet thrill nailbiter

Top 20 TF-IDF Features:


,term,tfidf,in_text
19,edgy,0.259726,True
26,instead get,0.258354,True
6,scream_neg,0.246803,True
27,top notch,0.232757,True
10,pass_neg,0.228349,True
18,notch,0.225240,True
25,special_neg effect_neg,0.224956,True
23,thrill,0.221709,True
7,feature_neg,0.214836,True
12,run_neg,0.198548,True


In [53]:
# Validate TF-IDF transformation consistency

sample_again = vectorizer.transform(
    X_train_text.iloc[[sample_idx]]
)

difference = sample_vector - sample_again

print("Number of different values:", difference.nnz)
print("TF-IDF transformation consistent:", difference.nnz == 0)

Number of different values: 0
TF-IDF transformation consistent: True


In [54]:
# Save the TF-IDF vectorizer
output_vectorizer_path = '../data/features/tfidf_vectorizer.pkl'

joblib.dump(
    vectorizer,
    output_vectorizer_path
)

print(f"TF-IDF vectorizer saved to: {output_vectorizer_path}")


TF-IDF vectorizer saved to: ../data/features/tfidf_vectorizer.pkl


In [55]:
# Save the TF-IDF sparse matrix

output_train_path = '../data/features/tfidf/X_train.npz'
output_test_path = '../data/features/tfidf/X_test.npz'

save_npz(output_train_path, X_train)
save_npz(output_test_path, X_test)

print(f"TF-IDF sparse matrices saved to: {output_train_path} and {output_test_path}")


TF-IDF sparse matrices saved to: ../data/features/tfidf/X_train.npz and ../data/features/tfidf/X_test.npz


In [56]:
# Create Metadata DataFrame (id_data, sentiment_label)

# Train Metadata
df_train_identity = df.loc[
    X_train_text.index,
    ['id_data', 'sentiment_label']
].copy()

# Test Metadata
df_test_identity = df.loc[
    X_test_text.index,
    ['id_data', 'sentiment_label']
].copy()

print("Train metadata:", df_train_identity.shape)
print("Test metadata :", df_test_identity.shape)

# Save the Metadata DataFrame
output_train_metadata_path = '../data/processed/train_metadata.csv'
output_test_metadata_path = '../data/processed/test_metadata.csv'

df_train_identity.to_csv(output_train_metadata_path, index=False)
df_test_identity.to_csv(output_test_metadata_path, index=False)

print(f"Metadata DataFrames saved to: {output_train_metadata_path} and {output_test_metadata_path}")

Train metadata: (35000, 2)
Test metadata : (15000, 2)
Metadata DataFrames saved to: ../data/processed/train_metadata.csv and ../data/processed/test_metadata.csv
